# E-CUP 2026 — первый CatBoost baseline

Ноутбук отвечает только за сценарий эксперимента. Загрузка данных, построение признаков, временная валидация, модель и отчёты находятся в `src/`. Это исключает расхождение логики между будущими экспериментами.

In [2]:
import sys, pyarrow
print(sys.executable)
print(pyarrow.__version__)

/Users/danasokol/.venvs/ozon-ecup/bin/python
25.0.1


In [3]:
from __future__ import annotations

from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd
import polars as pl

project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    FIRST_CATBOOST_ARTIFACT_DIR,
    FIRST_CATBOOST_SNAPSHOT_DIR,
    FIRST_CATBOOST_SUBMISSION_PATH,
    HORIZON_DAYS,
    TRAIN_PATH,
    ensure_output_dirs,
)
from src.data import all_users, latest_labeled_anchor, load_train, summarize_data, validate_user_days
from src.evaluation import feature_importance, save_json
from src.features import build_snapshot, save_snapshot
from src.models import final_iteration_count, make_final_model, make_validation_model, predict_gmv
from src.validation import anchors_available_for_holdout, feature_columns, make_historical_anchors, rmsle

ensure_output_dirs()
assert TRAIN_PATH.is_file(), f'Не найден файл: {TRAIN_PATH}'
print(f'Python: {platform.python_version()}')
print(f'Исходные данные: {TRAIN_PATH}')

Python: 3.12.2
Исходные данные: /Users/danasokol/Desktop/ML - соревы/OZON_GMV/data/train.parquet


## 1. Загрузка и контроль качества

Проверяем границы истории, согласованность ключевых денежных полей и отсутствие повторяющихся пар `user_id × event_date`.

In [4]:
data = load_train()
summary = summarize_data(data)
validate_user_days(data)
users = all_users(data)
max_date = summary['max_date']
latest_anchor = latest_labeled_anchor(data)

print(summary)
print('Число строк с пропусками по колонкам:')
print(data.null_count())
print(f'Последняя дата истории: {max_date}')
print(f'Последний якорь с доступным {HORIZON_DAYS}-дневным target: {latest_anchor}')

{'rows': 30631006, 'users': 250000, 'min_date': datetime.date(2025, 1, 1), 'max_date': datetime.date(2026, 2, 13), 'days': 409, 'positive_gmv_row_share': 0.15464418635156807, 'max_gmv_identity_error': 5.275069270282984e-11, 'max_order_identity_error': 0}
Число строк с пропусками по колонкам:
shape: (1, 18)
┌────────────┬─────────┬────────┬─────┬───┬─────────┬────────┬─────┬──────────┐
│ event_date ┆ user_id ┆ search ┆ cat ┆ … ┆ to_cart ┆ to_ord ┆ gmv ┆ searches │
│ ---        ┆ ---     ┆ ---    ┆ --- ┆   ┆ ---     ┆ ---    ┆ --- ┆ ---      │
│ u32        ┆ u32     ┆ u32    ┆ u32 ┆   ┆ u32     ┆ u32    ┆ u32 ┆ u32      │
╞════════════╪═════════╪════════╪═════╪═══╪═════════╪════════╪═════╪══════════╡
│ 0          ┆ 0       ┆ 0      ┆ 0   ┆ … ┆ 0       ┆ 0      ┆ 0   ┆ 0        │
└────────────┴─────────┴────────┴─────┴───┴─────────┴────────┴─────┴──────────┘
Последняя дата истории: 2026-02-13
Последний якорь с доступным 30-дневным target: 2026-01-14


## 2. Исторические срезы

`build_snapshot` из `src/features.py` формирует одну строку признаков на пользователя. Последний срез оставляем для честной временной проверки.

In [5]:
historical_anchors = make_historical_anchors(latest_anchor)
validation_anchor = historical_anchors[-1]
training_anchors = anchors_available_for_holdout(historical_anchors, validation_anchor)

print('Все исторические якоря:', historical_anchors)
print('Якоря для обучения в валидации:', training_anchors)
print('Валидационный якорь:', validation_anchor)

snapshots: dict = {} 
for anchor in historical_anchors:
    print(f'Строим срез {anchor} ...')
    snapshot = build_snapshot(data, users, anchor, with_target=True)
    snapshots[anchor] = snapshot
    path = save_snapshot(snapshot, FIRST_CATBOOST_SNAPSHOT_DIR, anchor, 'train')
    positive_share = snapshot.select((pl.col('target') > 0).mean()).item()
    print(f'  shape={snapshot.shape}, доля положительных target={positive_share:.4f}, сохранено: {path.name}')

train_valid = pl.concat([snapshots[anchor] for anchor in training_anchors], how='vertical_relaxed')
valid = snapshots[validation_anchor]
feature_cols = feature_columns(train_valid)

print(f'Число признаков: {len(feature_cols)}')
print(f'Обучающих строк: {train_valid.height:,}; валидационных строк: {valid.height:,}')

Все исторические якоря: [datetime.date(2025, 7, 2), datetime.date(2025, 7, 30), datetime.date(2025, 8, 27), datetime.date(2025, 9, 24), datetime.date(2025, 10, 22), datetime.date(2025, 11, 19), datetime.date(2025, 12, 17), datetime.date(2026, 1, 14)]
Якоря для обучения в валидации: [datetime.date(2025, 7, 2), datetime.date(2025, 7, 30), datetime.date(2025, 8, 27), datetime.date(2025, 9, 24), datetime.date(2025, 10, 22), datetime.date(2025, 11, 19)]
Валидационный якорь: 2026-01-14
Строим срез 2025-07-02 ...
  shape=(250000, 101), доля положительных target=0.4586, сохранено: train_2025-07-02.parquet
Строим срез 2025-07-30 ...
  shape=(250000, 101), доля положительных target=0.4890, сохранено: train_2025-07-30.parquet
Строим срез 2025-08-27 ...
  shape=(250000, 101), доля положительных target=0.4947, сохранено: train_2025-08-27.parquet
Строим срез 2025-09-24 ...
  shape=(250000, 101), доля положительных target=0.5162, сохранено: train_2025-09-24.parquet
Строим срез 2025-10-22 ...
  shape=

## 3. Валидационный CatBoost

Сравниваем CatBoost с наивным прогнозом — GMV пользователя за предыдущие 30 дней.

In [6]:
X_train = train_valid.select(feature_cols).to_pandas()
y_train = np.log1p(train_valid['target'].to_numpy())
X_valid = valid.select(feature_cols).to_pandas()
y_valid = valid['target'].to_numpy()

baseline_pred = valid['gmv_sum_30d'].to_numpy()
baseline_score = rmsle(y_valid, baseline_pred)

model = make_validation_model()
model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, np.log1p(y_valid)),
    early_stopping_rounds=200,
    use_best_model=True,
)

valid_pred = predict_gmv(model, X_valid)
catboost_score = rmsle(y_valid, valid_pred)

print(f'RMSLE наивного autoregressive baseline: {baseline_score:.6f}')
print(f'RMSLE CatBoost:                       {catboost_score:.6f}')
print(f'Лучшая итерация: {model.get_best_iteration()}')

importance = feature_importance(model, feature_cols)
display(importance.head(25))
importance.to_csv(FIRST_CATBOOST_ARTIFACT_DIR / 'feature_importance_validation.csv', index=False)

metrics = {
    'validation_anchor': validation_anchor.isoformat(),
    'baseline_rmsle': baseline_score,
    'catboost_rmsle': catboost_score,
    'best_iteration': int(model.get_best_iteration()),
    'n_features': len(feature_cols),
    'n_train_rows_for_validation': int(train_valid.height),
}
save_json(metrics, FIRST_CATBOOST_ARTIFACT_DIR / 'validation_metrics.json')

0:	learn: 2.2950005	test: 2.2395117	best: 2.2395117 (0)	total: 471ms	remaining: 14m 8s
200:	learn: 1.7255530	test: 1.7259396	best: 1.7259396 (200)	total: 1m 31s	remaining: 12m 5s
400:	learn: 1.7204125	test: 1.7239255	best: 1.7238200 (392)	total: 3m 6s	remaining: 10m 50s
600:	learn: 1.7167445	test: 1.7228342	best: 1.7228311 (580)	total: 4m 34s	remaining: 9m 7s
800:	learn: 1.7137632	test: 1.7227972	best: 1.7225883 (677)	total: 5m 51s	remaining: 7m 18s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 1.722588307
bestIteration = 677

Shrink model to first 678 iterations.
RMSLE наивного autoregressive baseline: 2.195065
RMSLE CatBoost:                       1.722584
Лучшая итерация: 677


,feature,importance
72,to_ord_active_days_90d,10.824424
53,to_ord_sum_90d,10.645500
0,observed_active_days_180d,7.723856
69,gmv_active_days_90d,7.010638
73,days_since_gmv,6.921608
74,days_since_to_ord,5.959817
54,gmv_sum_90d,5.447725
3,searches_sum_7d,2.690221
31,gmv_search_sum_30d,2.678263
61,gmv_mean_active_90d,2.622917


## 4. Финальная модель и submission

Финальная модель использует все размеченные исторические срезы. Признаки финального среза строятся на последней дате истории без доступа к будущему GMV.

In [7]:
final_train = pl.concat([snapshots[anchor] for anchor in historical_anchors], how='vertical_relaxed')
final_snapshot = build_snapshot(data, users, max_date, with_target=False)
save_snapshot(final_snapshot, FIRST_CATBOOST_SNAPSHOT_DIR, max_date, 'test')

X_final_train = final_train.select(feature_cols).to_pandas()
y_final_train = np.log1p(final_train['target'].to_numpy())
X_test = final_snapshot.select(feature_cols).to_pandas()

final_model = make_final_model(final_iteration_count(model.get_best_iteration()))
final_model.fit(X_final_train, y_final_train)
test_pred = predict_gmv(final_model, X_test)

submission = pd.DataFrame({
    'user_id': final_snapshot['user_id'].to_numpy(),
    'predict': np.clip(test_pred, 0, None),
})
assert submission.shape == (users.height, 2)
assert submission['user_id'].is_unique
assert (submission['predict'] >= 0).all()

submission.to_csv(FIRST_CATBOOST_SUBMISSION_PATH, index=False, float_format='%.8f')
final_model.save_model(FIRST_CATBOOST_ARTIFACT_DIR / 'catboost_log_gmv.cbm')
save_json({'feature_columns': feature_cols}, FIRST_CATBOOST_ARTIFACT_DIR / 'feature_columns.json')

print(f'Сабмит сохранён: {FIRST_CATBOOST_SUBMISSION_PATH}')
print(f'Строк: {len(submission):,}')
print(submission['predict'].describe(percentiles=[0.5, 0.9, 0.99]))
display(submission.head())

0:	learn: 2.2903159	total: 548ms	remaining: 7m 4s
200:	learn: 1.7282016	total: 2m 33s	remaining: 7m 19s
400:	learn: 1.7237445	total: 4m 51s	remaining: 4m 33s
600:	learn: 1.7206272	total: 6m 50s	remaining: 2m
776:	learn: 1.7184220	total: 8m 32s	remaining: 0us
Сабмит сохранён: /Users/danasokol/Desktop/ML - соревы/OZON_GMV/submissions/submission_first_catboost.csv
Строк: 250,000
count    250000.000000
mean         34.686270
std          91.308591
min           0.000000
50%           6.690437
90%          87.667551
99%         410.353115
max        3248.050141
Name: predict, dtype: float64


,user_id,predict
0,2,1.891022
1,7,106.086748
2,15,13.724067
3,18,119.038063
4,23,0.537716
